# EgoEvGesture — Independent Validation of the Published Heterogeneous Split v1

**Objective:** Verify the **Kaggle Dataset containing CSVs exported by the previous split notebook** and cross-check it against the original event dataset. This notebook **does not recreate splits, upload data, read all 207 GB, or modify `/kaggle/input`**.

**Required Kaggle Inputs:**

1. **Raw:** `hoanganh94/egoevgesture-raw` (5,398 event files; masks and labels are not counted as event samples).
2. **Split:** `EgoEvGesture Heterogeneous Cleaned Splits v1`, created from the split notebook output, containing `train.csv`, `val.csv`, `test.csv`, `split_manifest.json`, and `verified_internal_duplicates.csv`.

**Expected cleaned heterogeneous split:** Train Subjects 3–7 (**2,626**), Validation Subject 8 (**489**), Test Subjects 1, 2, 9 (**1,695**). Exclude 587 Subject 10 duplicates and one internal exact duplicate, leaving **4,810** samples.

**Important:** This validation assignment was adapted to handle duplicates; it is **not** an official published validation split from the EgoEvGesture authors. The split dataset stores indices, not event `.npy` files.

## 1 — Automatically discover the two Kaggle Inputs

Run top-to-bottom or choose **Run All**. The notebook supports both `/kaggle/input/datasets/hoanganh94/...` and shortened Kaggle mount paths. If the split dataset slug changes, discovery still uses `split_manifest.json`. If multiple candidates are found, set `SPLIT_ROOT_HINT` in the configuration cell.

In [ ]:
from pathlib import Path, PurePosixPath
from collections import defaultdict
import os, re, json, hashlib, time, math
import numpy as np
import pandas as pd

INPUT_BASE = Path(os.environ.get("EGOEV_INPUT_BASE", "/kaggle/input"))
RAW_ROOT_HINT = os.environ.get("EGOEV_RAW_ROOT", "/kaggle/input/datasets/hoanganh94/egoevgesture-raw")
SPLIT_ROOT_HINT = os.environ.get("EGOEV_SPLIT_ROOT", "")  # If needed, paste the new split dataset Copy Path here.
OUT_DIR = Path(os.environ.get("EGOEV_VALIDATION_OUTPUT", "/kaggle/working/egoev_split_dataset_validation"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Minimize I/O by hashing complete files only for matching fingerprint candidates.
VERIFY_SELECTED_EXACT_DUPLICATES = True
# By default inspect four sample headers per split; do not load full event streams into RAM.
HEADER_SAMPLES_PER_SPLIT = 4
# By default do NOT rehash all 587 Subject 5/10 pairs (covered in the original audit).
SUBJ5_10_SHA_PAIRS_PER_CAMERA = 0

EXPECTED_SUBJECTS = {"train": {3,4,5,6,7}, "val": {8}, "test": {1,2,9}}
EXPECTED_COUNTS = {"train": 2626, "val": 489, "test": 1695}
EXPECTED_RAW = 5398
EXPECTED_EXCLUDED_SUBJ10 = 587
EXPECTED_EXTRA_EXCLUDED = 1
REQUIRED_CSV = ["train.csv","val.csv","test.csv","all_splits.csv","excluded_samples.csv",
                "original_event_index.csv","class_distribution.csv","subject_distribution.csv",
                "verified_internal_duplicates.csv"]
CHECKS = []
def record(name, okay, detail="", level="ERROR"):
    ok = bool(okay)
    CHECKS.append({"check":name, "passed":ok, "level":level if not ok else "OK", "detail":str(detail)})
    print(("PASS" if ok else "FAIL"), "|", name, ("| "+str(detail)) if detail else "")
    return ok

def list_input_tree(base=INPUT_BASE, depth_limit=4):
    if not base.exists(): return [f"Does not exist: {base}"]
    output=[]
    for root, dirs, files in os.walk(base):
        relative=Path(root).relative_to(base)
        dep=len(relative.parts)
        if dep > depth_limit:
            dirs[:]=[]; continue
        if relative.parts and (re.fullmatch(r"subj\d+_(?:left|right)",relative.parts[-1]) or relative.parts[-1] in {"mask","label"}):
            dirs[:]=[]; continue
        output.append("  "*dep + (relative.name if dep else str(base)) + "/")
        if dep == depth_limit: dirs[:]=[]
        else: dirs[:]=sorted(d for d in dirs if not d.startswith('.'))
        if len(output)>=100: break
    return output

print("=== AVAILABLE MOUNTS ===")
print("\n".join(list_input_tree()))

def discover_split_roots(base):
    found=[]
    for root, dirs, files in os.walk(base):
        p=Path(root)
        depth=len(p.relative_to(base).parts)
        if "split_manifest.json" in files and {"train.csv","val.csv","test.csv"}.issubset(files):
            found.append(p)
            dirs[:]=[]
            continue
        if depth>=6 or re.fullmatch(r"subj\d+_(?:left|right)",p.name) or p.name in {"mask","label"}:
            dirs[:]=[]
        else:
            dirs[:]=sorted(d for d in dirs if not d.startswith('.'))
    return sorted(set(found))

raw_hint=Path(RAW_ROOT_HINT)
if raw_hint.is_dir():
    RAW_ROOT=raw_hint
else:
    raw_options=[]
    for root, dirs, files in os.walk(INPUT_BASE):
        d=Path(root)
        if (d/'subj1_left').is_dir() and (d/'subj10_right').is_dir():
            raw_options.append(d)
            dirs[:]=[]
        elif len(d.relative_to(INPUT_BASE).parts)>=5:
            dirs[:]=[]
    if len(raw_options)!=1:
        raise FileNotFoundError("Could not uniquely identify the raw dataset. Found: " + str(raw_options) + ". Add the original dataset as Input.")
    RAW_ROOT=raw_options[0]

if SPLIT_ROOT_HINT:
    SPLIT_ROOT=Path(SPLIT_ROOT_HINT)
    if not SPLIT_ROOT.is_dir():
        raise FileNotFoundError("SPLIT_ROOT_HINT does not exist: " + str(SPLIT_ROOT))
else:
    split_candidates=discover_split_roots(INPUT_BASE)
    if len(split_candidates)!=1:
        raise FileNotFoundError("Could not uniquely identify the attached split dataset. Candidates: " + str(split_candidates) + ". Add the published CSV dataset as Input or set SPLIT_ROOT_HINT.")
    SPLIT_ROOT=split_candidates[0]

print("\nRAW_ROOT:",RAW_ROOT)
print("SPLIT_ROOT (reading the PUBLISHED dataset):",SPLIT_ROOT)
print("NEW REPORT DIRECTORY:",OUT_DIR)
assert RAW_ROOT.resolve() != SPLIT_ROOT.resolve(), 'Raw and split datasets must have different roots'
assert SPLIT_ROOT.is_relative_to(INPUT_BASE), 'The split dataset must be mounted under /kaggle/input (not prior notebook Output)'


## 2 — Verify the published manifest and CSV checksums

Compare SHA-256 for the **nine small CSV files** against `csv_sha256` in `split_manifest.json`. This catches missing, modified, or mixed-version files before training.

In [ ]:
manifest_file=SPLIT_ROOT/"split_manifest.json"
if not manifest_file.is_file():
    raise FileNotFoundError(f"Missing {manifest_file}. Kaggle may have published CSVs without the manifest.")
manifest=json.loads(manifest_file.read_text(encoding="utf-8"))

def sha256_file(path,block=8*1024*1024):
    h=hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b=f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

record("Schema manifest", manifest.get("schema_version")=="egoev-clean-split/1.0",manifest.get("schema_version"))
record("Source dataset in manifest",manifest.get("source_ref")=="hoanganh94/egoevgesture-raw",manifest.get("source_ref"))
record("Heterogeneous protocol",manifest.get("protocol")=="heterogeneous_subject_holdout_cleaned",manifest.get("protocol"))
for split in ("train","val","test"):
    field=split+"_subjects"
    record("Manifest subject "+split,set(manifest.get(field,[]))==EXPECTED_SUBJECTS[split],str(manifest.get(field)))
record("Excluded subject in manifest",manifest.get("excluded_duplicate_subject")==10,manifest.get("excluded_duplicate_subject"))

stored_hashes=manifest.get("csv_sha256",{})
for name in REQUIRED_CSV:
    file=SPLIT_ROOT/name
    if not file.is_file():
        record("Required file: "+name,False,"MISSING")
        continue
    actual=sha256_file(file)
    expected=stored_hashes.get(name)
    record("SHA256 CSV: "+name,expected is not None and actual==expected,
           f"expected={str(expected)[:14]}... actual={actual[:14]}..." if expected != actual else "match")
if any(not (SPLIT_ROOT/n).is_file() for n in REQUIRED_CSV):
    raise FileNotFoundError("Required CSV files missing; inspect FAIL findings above and verify the Dataset was published from Output.")


## 3 — Validate every row in train/validation/test

Check expected subjects, `gesture_id = action_raw_id % 38`, camera side, recording condition, filename, safe relative paths, class coverage, and disjoint splits. Cross-check each row against `original_event_index.csv` and `all_splits.csv`.

In [ ]:
columns=["relative_path","subject","camera","group","action_raw_id","trial_id","gesture_id","recording_condition","size_bytes","split"]
int_cols=["subject","action_raw_id","trial_id","gesture_id","recording_condition","size_bytes"]
text_cols=["relative_path","camera","group","split"]

def read_typed_csv(filename, required_cols=None):
    frame=pd.read_csv(SPLIT_ROOT/filename,dtype="string",keep_default_na=False)
    if required_cols is not None:
        missing=set(required_cols)-set(frame.columns)
        if missing: raise ValueError(f"{filename}: missing columns {sorted(missing)}")
    return frame

def coerce_nums(frame,names,filename):
    for c in names:
        if c not in frame: continue
        try:
            vals=pd.to_numeric(frame[c],errors="raise")
            if vals.isna().any() or (vals%1!=0).any():
                raise ValueError("null or non-integer value")
            frame[c]=vals.astype("int64")
        except Exception as e:
            raise ValueError(f"Column {c} trong {filename} is not an integer: {e}")
    return frame

parts={s:coerce_nums(read_typed_csv(f"{s}.csv",columns),int_cols,f"{s}.csv") for s in EXPECTED_SUBJECTS}
for split,df in parts.items():
    record("Sample count for "+split,len(df)==EXPECTED_COUNTS[split],f"{len(df)} samples")
    record("Split values in "+split,set(df.split)=={split},str(df.split.unique().tolist()))
    record("Subject assignment in "+split,set(df.subject)==EXPECTED_SUBJECTS[split],sorted(set(df.subject)))
    record("All 38 classes in "+split,set(df.gesture_id)==set(range(38)),f"{df.gesture_id.nunique()} classes")

selected=pd.concat([parts[s] for s in ("train","val","test")],ignore_index=True)
record("Selected sample count",len(selected)==4810,len(selected))
record("No duplicate paths across the three splits",selected.relative_path.is_unique,f"unique={selected.relative_path.nunique()}")
record("Each subject belongs to exactly one split",selected.groupby("subject").split.nunique().max()==1)
record("Correct gesture-label formula",(selected.gesture_id==selected.action_raw_id%38).all())
record("Correct recording condition",(selected.recording_condition==selected.action_raw_id//38).all())
record("Valid action/trial IDs",selected.action_raw_id.between(0,75).all() and selected.trial_id.ge(0).all())
record("Only left/right camera labels",set(selected.camera).issubset({"left","right"}))

pattern=re.compile(r"^subj(\d+)_(left|right)/act(\d+)_(\d+)\.npy$")
def row_path_valid(r):
    path=str(r.relative_path)
    if path.startswith("/") or "\\" in path or ".." in PurePosixPath(path).parts:
        return False
    m=pattern.fullmatch(path)
    return bool(m and int(m[1])==r.subject and m[2]==r.camera and int(m[3])==r.action_raw_id
                and int(m[4])==r.trial_id and r.group==f"subj{r.subject}_{r.camera}")
bad_paths=[r.relative_path for r in selected.itertuples(index=False) if not row_path_valid(r)]
record("Paths, subjects, cameras, and filenames are consistent",not bad_paths,f"errors: {bad_paths[:5]}")

all_splits=coerce_nums(read_typed_csv("all_splits.csv",columns),int_cols,"all_splits.csv")
def compare_rows(a,b,cols):
    try:
        aa=a[cols].sort_values(cols,kind="stable").reset_index(drop=True)
        bb=b[cols].sort_values(cols,kind="stable").reset_index(drop=True)
        aa.columns.name=None; bb.columns.name=None
        aa.index.name=None; bb.index.name=None
        pd.testing.assert_frame_equal(aa,bb,check_dtype=True)
        return True,"All rows match"
    except (AssertionError,KeyError,ValueError) as e:
        return False,str(e)[:300]
ok,detail=compare_rows(selected,all_splits,columns)
record("all_splits.csv == concatenated train/val/test",ok,detail)
record("Manifest sample counts agree with CSVs",manifest.get("sample_counts",{})=={k:len(v) for k,v in parts.items()},manifest.get("sample_counts"))
record("Manifest class counts agree with CSVs",manifest.get("class_counts",{})=={k:int(v.gesture_id.nunique()) for k,v in parts.items()})


## 4 — Reconcile the original index, exclusions, and class distributions

Verify **5,398 original = 4,810 retained + 588 excluded**: 587 samples from Subject 10 and one internal exact duplicate. Confirm the published distribution CSVs agree with the three selected splits.

In [ ]:
source_index=coerce_nums(read_typed_csv("original_event_index.csv",columns[:-1]+["split","exclude_reason"]),int_cols,"original_event_index.csv")
excluded=coerce_nums(read_typed_csv("excluded_samples.csv",["relative_path","subject","split","exclude_reason"]),["subject"],"excluded_samples.csv")
record("Original index contains exactly 5,398 events",len(source_index)==EXPECTED_RAW,len(source_index))
record("Original index has unique paths",source_index.relative_path.is_unique)
record("All original entries accounted for",set(source_index.relative_path)==set(selected.relative_path)|set(excluded.relative_path))
record("Selected and excluded paths are disjoint",not (set(selected.relative_path)&set(excluded.relative_path)))
record("Excluded list has unique paths",excluded.relative_path.is_unique)
record("Exactly 588 samples excluded",len(excluded)==588,len(excluded))
record("Exactly 587 Subject 10 samples excluded",len(excluded[excluded.subject==10])==EXPECTED_EXCLUDED_SUBJ10,
       len(excluded[excluded.subject==10]))
record("Exactly one internal duplicate excluded",(excluded.exclude_reason=="duplicate_within_split").sum()==EXPECTED_EXTRA_EXCLUDED,
       int((excluded.exclude_reason=="duplicate_within_split").sum()))
record("Subject 10 exclusion reason",excluded.loc[excluded.subject==10,"exclude_reason"].eq("duplicate_of_subject5").all())
record("All excluded entries have split=excluded",excluded.split.eq("excluded").all())
source_selected=source_index[source_index.split.isin(["train","val","test"])].copy()
ok,detail=compare_rows(selected,source_selected,columns)
record("Original index agrees with the three splits",ok,detail)

ct=pd.crosstab(selected.gesture_id,selected.split).reindex(index=range(38),columns=["train","val","test"],fill_value=0)
ct.index.name="gesture_id"
ct["total"]=ct.sum(axis=1)
actual_class=ct.reset_index()
published_class=coerce_nums(read_typed_csv("class_distribution.csv",actual_class.columns),list(actual_class.columns),"class_distribution.csv")
ok,detail=compare_rows(actual_class,published_class,list(actual_class.columns))
record("class_distribution.csv matches",ok,detail)

st=pd.crosstab(selected.subject,selected.split).reindex(columns=["train","val","test"],fill_value=0)
st.index.name="subject"
actual_subject=st.reset_index()
published_subject=coerce_nums(read_typed_csv("subject_distribution.csv",actual_subject.columns),list(actual_subject.columns),"subject_distribution.csv")
ok,detail=compare_rows(actual_subject,published_subject,list(actual_subject.columns))
record("subject_distribution.csv matches",ok,detail)

print("\n=== SUBJECT STATISTICS ===")
print(actual_subject.to_string(index=False))
print("\n=== CLASS DISTRIBUTION — FIRST / LAST FIVE ===")
print(pd.concat([actual_class.head(5),actual_class.tail(5)]).to_string(index=False))


## 5 — Identify the **actual internal duplicate pair** and recompute SHA-256

Print the **retained path** and **excluded path**, class, split, byte count, and SHA-256. Then independently hash **both actual files** from the internal-duplicate cluster and compare their digests with the published record. Do not infer the second filename from the first.

In [ ]:
dup=read_typed_csv("verified_internal_duplicates.csv",
    ["relative_path","sha256","split","gesture_id","removed_as_duplicate"])
coerce_nums(dup,["gesture_id"],"verified_internal_duplicates.csv")
dup["removed_as_duplicate_bool"]=dup.removed_as_duplicate.str.lower().map({"true":True,"false":False})
record("Valid removed_as_duplicate column",dup.removed_as_duplicate_bool.notna().all())
record("Exactly one internal duplicate cluster",dup.sha256.nunique()==1 and len(dup)==2,
       f"{dup.sha256.nunique()} clusters; {len(dup)} members")
record("Exactly one entry excluded from the cluster",int(dup.removed_as_duplicate_bool.fillna(False).sum())==1)
record("Duplicate pair has the same class and split",dup.gesture_id.nunique()==1 and dup.split.nunique()==1)
for r in dup.itertuples(index=False):
    contained=(r.relative_path in set(excluded.relative_path) if r.removed_as_duplicate_bool
               else r.relative_path in set(selected.relative_path))
    record("Retained/excluded status: "+r.relative_path,contained, "removed="+str(r.removed_as_duplicate_bool))

print("\n=== ACTUAL INTERNAL DUPLICATE PAIR ===")
print(dup[["relative_path","sha256","split","gesture_id","removed_as_duplicate"]].to_string(index=False))
if len(dup)==2:
    print("\nRETAINED file:",dup.loc[~dup.removed_as_duplicate_bool,"relative_path"].tolist())
    print("EXCLUDED file:",dup.loc[dup.removed_as_duplicate_bool,"relative_path"].tolist())

for r in dup.itertuples(index=False):
    event_file=RAW_ROOT/r.relative_path
    if not event_file.is_file():
        record("Read duplicate-pair file: "+r.relative_path,False,"File not found")
        continue
    actual_digest=sha256_file(event_file)
    record("Recomputed SHA-256: "+r.relative_path,actual_digest==r.sha256,
           "digest matches" if actual_digest==r.sha256 else "DIGEST MISMATCH")


## 6 — Cross-check paths, sizes, and sampled headers against raw data

Call `stat()` on the published source index without reading complete file contents. Inspect a representative sample of `.npy` headers via memory mapping; this does not establish full payload integrity. If raw data has not been added as Input, the notebook stops at step 1.

In [ ]:
missing=[]; mismatched=[]
for j,r in enumerate(source_index.itertuples(index=False),1):
    p=RAW_ROOT/r.relative_path
    if not p.is_file():
        missing.append(r.relative_path)
    else:
        actual_size=p.stat().st_size
        if actual_size != int(r.size_bytes):
            mismatched.append((r.relative_path,int(r.size_bytes),actual_size))
    if j%1000==0: print(f"Checking file: {j}/{len(source_index)}")
record("No raw event files are missing",not missing,f"missing={len(missing)} {missing[:3]}")
record("Event file sizes match the index",not mismatched,f"wrong={len(mismatched)} {mismatched[:3]}")

header_rows=[]
for split,df in parts.items():
    chosen=df.sort_values("relative_path").head(HEADER_SAMPLES_PER_SPLIT)
    for r in chosen.itertuples(index=False):
        try:
            obj=np.load(RAW_ROOT/r.relative_path,mmap_mode="r",allow_pickle=False)
            header_rows.append({"split":split,"relative_path":r.relative_path,"shape":str(tuple(obj.shape)),
                "dtype":str(obj.dtype),"is_nx4":bool(obj.ndim==2 and obj.shape[1]==4),"error":""})
            del obj
        except Exception as exc:
            header_rows.append({"split":split,"relative_path":r.relative_path,"shape":"","dtype":"",
                                "is_nx4":False,"error":str(exc)})
headers_df=pd.DataFrame(header_rows)
record("Sampled headers are readable and have Nx4 shape",headers_df.error.eq("").all() and headers_df.is_nx4.all(),
       f"{len(headers_df)} sampled files")
print(headers_df.to_string(index=False))
headers_df.to_csv(OUT_DIR/"header_sample_check.csv",index=False)


## 7 — Independently check for **exact-byte leakage** across splits

Among the **4,810 retained files**, filter by file size, compute head/middle/tail fingerprints, then compute complete SHA-256 only when those fingerprints match. This independently checks the split-creation result. It does not detect near-duplicates with different bytes, and it does not read all 207 GB of source data.

In [ ]:
def sample_fingerprint(path,size,chunk=256*1024):
    h=hashlib.sha256(b'EgoEv-sample-v1'+str(size).encode('ascii'))
    length=min(chunk,size)
    offsets=sorted({0,max(0,(size-length)//2),max(0,size-length)})
    with path.open('rb') as f:
        for off in offsets:
            f.seek(off)
            b=f.read(length)
            if len(b)!=length: raise IOError(f"Short read for {path}")
            h.update(off.to_bytes(8,'little')); h.update(b)
    return h.hexdigest()

exact_dup_after=[]
if VERIFY_SELECTED_EXACT_DUPLICATES:
    size_freq=selected.size_bytes.value_counts()
    candidates=selected[selected.size_bytes.map(size_freq).gt(1)].copy()
    print("Equal-size candidates:",len(candidates))
    fp_entries=[]
    for r in candidates.itertuples(index=False):
        fp_entries.append({"relative_path":r.relative_path,"size_bytes":r.size_bytes,
                           "fingerprint":sample_fingerprint(RAW_ROOT/r.relative_path,int(r.size_bytes))})
    fp=pd.DataFrame(fp_entries,columns=["relative_path","size_bytes","fingerprint"])
    if len(fp):
        freq=fp.groupby(["size_bytes","fingerprint"]).relative_path.transform('size')
        sha_candidates=fp[freq>1].copy()
    else: sha_candidates=fp
    print("Full SHA-256 required for:",len(sha_candidates),"file /",round(sha_candidates.size_bytes.sum()/1e9,3),"GB")
    verified=[]
    for r in sha_candidates.itertuples(index=False):
        verified.append({"relative_path":r.relative_path,"size_bytes":r.size_bytes,
                         "sha256":sha256_file(RAW_ROOT/r.relative_path)})
    sha=pd.DataFrame(verified,columns=["relative_path","size_bytes","sha256"])
    if len(sha):
        for _,g in sha.groupby(["size_bytes","sha256"]):
            if len(g)>=2:
                pairs=g.relative_path.tolist()
                matched=selected[selected.relative_path.isin(pairs)]
                exact_dup_after.append({"paths":pairs,"splits":sorted(matched.split.unique().tolist()),
                                        "labels":sorted(matched.gesture_id.unique().tolist()),"sha256":g.sha256.iloc[0]})
    record("No byte-identical duplicates remain among 4,810 retained samples",len(exact_dup_after)==0,
           f"remaining duplicate clusters={len(exact_dup_after)}")
    print("Remaining duplicate clusters:",exact_dup_after[:5])
else:
    record("Exact duplicate verification enabled",False,"VERIFY_SELECTED_EXACT_DUPLICATES=False: insufficient to finalize the dataset")


## 8 — Summarize results and export a new validation report (read-only)

New reports are saved under `/kaggle/working/egoev_split_dataset_validation`. If every check reports `PASS`, the attached published split can be reused for subsequent experiments. On `FAIL`, inspect `00_findings.csv` and the error messages. Do not change or delete the raw dataset.

In [ ]:
findings=pd.DataFrame(CHECKS)
findings.to_csv(OUT_DIR/"00_findings.csv",index=False)
failed=findings[~findings.passed].copy()
summary={
 "source_dataset":str(RAW_ROOT), "published_split_dataset":str(SPLIT_ROOT),
 "expected_counts":EXPECTED_COUNTS,"actual_counts":{k:len(v) for k,v in parts.items()},
 "original_event_samples":len(source_index),"kept":len(selected),"excluded":len(excluded),
 "excluded_subject10":int((excluded.subject==10).sum()),
 "internal_duplicate_removed":int((excluded.exclude_reason=="duplicate_within_split").sum()),
 "pair_internal_duplicate_paths":dup.relative_path.tolist(),
 "checks_total":int(len(findings)), "checks_passed":int(findings.passed.sum()),
 "checks_failed":int(len(failed)),"all_pass":len(failed)==0,
 "note":"Read-only validation of split CSVs and event paths. SHA for same-size/sampled-fingerprint matches, not near-duplicate semantic screening."
}
(OUT_DIR/"00_validation_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
lines=["# EgoEvGesture — published Heterogeneous Split v1 verification", "",
       f"- Split dataset mount: `{SPLIT_ROOT}`", f"- Raw dataset mount: `{RAW_ROOT}`",
       f"- Expected: train/val/test = {EXPECTED_COUNTS}",
       f"- Actual: {summary['actual_counts']}",
       f"- Original event samples: {summary['original_event_samples']}",
       f"- Kept: {summary['kept']}; excluded: {summary['excluded']}",
       f"- SHA duplicate evidence: {summary['pair_internal_duplicate_paths']}",
       f"- Checks: {summary['checks_passed']}/{summary['checks_total']} passed", "",
       "## Checks failed", ""]
if failed.empty: lines.append("None")
else:
    for r in failed.itertuples(index=False):
        lines.append(f"- **{r.check}**: {r.detail}")
lines.extend(["","## Scope and limitations","","CSV checksum verification and exact-duplicate candidate filtering performed. Does not prove no near-duplicates or identical gestures recorded differently. Validation/test are research-defined, not authors' official full split."])
(OUT_DIR/"00_validation_report.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
print("="*70)
print("VALIDATION SUMMARY FOR THE ATTACHED PUBLISHED SPLIT")
print("="*70)
print("SPLIT:",SPLIT_ROOT)
print("TRAIN / VAL / TEST:",summary["actual_counts"])
print("ORIGINAL / RETAINED / EXCLUDED:",len(source_index),len(selected),len(excluded))
print("INTERNAL DUPLICATE PAIR:",summary["pair_internal_duplicate_paths"])
print("PASS:",summary["checks_passed"],"/",summary["checks_total"])
if failed.empty:
    print("CONCLUSION: PASS — published Kaggle Dataset CSVs are consistent and reusable.")
else:
    print("CONCLUSION: FAIL — validation issues found:")
    print(failed[["check","detail"]].to_string(index=False))
print("REPORT:",OUT_DIR/"00_validation_report.md")
print("FINDINGS:",OUT_DIR/"00_findings.csv")
print("JSON:",OUT_DIR/"00_validation_summary.json")
if not failed.empty:
    raise RuntimeError("Dataset split validation FAILED. Inspect the reports; training is not recommended yet.")


## 9 — Read a published training example (no manual path needed)

The next cell uses `RAW_ROOT` and `SPLIT_ROOT`, which were **automatically discovered in the first configuration cell**. There is no need to paste the Kaggle Copy Path.

**Execution order:** Add Input for both raw and split datasets, then choose **Run All**. After the preceding validations pass, this cell loads the header of one training `.npy` sample using `mmap_mode='r'`, without loading the whole event array into RAM.

If you execute this cell alone in a new session, first run the initial configuration cell to initialize the path variables.

In [ ]:
# Read one sample without pasting paths or re-uploading the dataset.
from pathlib import Path
import pandas as pd
import numpy as np

if 'RAW_ROOT' not in globals() or 'SPLIT_ROOT' not in globals():
    raise RuntimeError('Run cell 2 (dataset discovery) before this sample-read cell.')

train_csv = Path(SPLIT_ROOT) / 'train.csv'
if not train_csv.is_file():
    raise FileNotFoundError(f'Could not find train.csv in the attached split dataset: {train_csv}')
train_df = pd.read_csv(train_csv)
if train_df.empty:
    raise RuntimeError('train.csv is empty')

sample = train_df.iloc[0]
event_path = Path(RAW_ROOT) / str(sample['relative_path'])
if not event_path.is_file():
    raise FileNotFoundError(f'CSV path does not exist in the raw dataset: {event_path}')

arr = np.load(event_path, mmap_mode='r', allow_pickle=False)
print('RAW_ROOT:', RAW_ROOT)
print('SPLIT_ROOT:', SPLIT_ROOT)
print('Train samples:', len(train_df))
print('Sample relative path:', sample['relative_path'])
print('Shape / dtype:', arr.shape, arr.dtype)
print('Gesture label:', int(sample['gesture_id']))
assert arr.ndim == 2 and arr.shape[1] == 4, 'Training sample does not have Nx4 shape'
print('SAMPLE READ: PASS')
